In [1]:
import pandas as pd
import numpy as np
import os
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

# --- CONFIGURATION ---
ALL_COHORT_STEMS = [
    'Public_study__FengQ_2015', 'Public_study__GuptaA_2019', 'Public_study__LiuNN_2022',
    'Public_study__ThomasAM_2018b', 'Public_study__VogtmannE_2016', 'Public_study__WirbelJ_2018',
    'Public_study__YachidaS_2019', 'Public_study__YangJ_2020', 'Public_study__YangY_2021',
    'Public_study__YuJ_2015', 'Public_study__ZellerG_2014', 'This_study_cohort1__ONCOBIOME_AtezoTRIBE',
    'This_study_cohort2__ONCOBIOME_COLOBIOME', 'This_study_cohort3__ONCOBIOME_IIGM_CZ',
    'This_study_cohort4__ONCOBIOME_IIGM_IT', 'This_study_cohort5__NSHII', 'This_study_cohort6__IIGM_TU'
]
TRAINING_ONLY_COHORT = 'This_study_cohort5__NSHII'

In [ ]:
def prepare_single_cohort(cohort_stem, data_type='functions', base_data_dir='../../data'):
    """
    Prepares a single cohort's data.
    
    Parameters:
    -----------
    cohort_stem : str
        Cohort identifier (e.g., 'Public_study__FengQ_2015')
    data_type : str
        Type of data: 'functions', 'taxa', or 'combined'
    base_data_dir : str
        Base directory containing data subdirectories
    
    Returns:
    --------
    X : pd.DataFrame or None
        Feature matrix (samples x features) or None if error
    y : pd.Series or None
        Target labels (0 = CRC, 1 = Control) or None if error
    """
    try:
        # Determine file paths based on data type
        if data_type == 'functions':
            input_file = os.path.join(base_data_dir, 'pathways', f'{cohort_stem}.tsv')
        elif data_type == 'taxa':
            input_file = os.path.join(base_data_dir, 'taxa', f'{cohort_stem}.tsv')
        elif data_type == 'combined':
            # For combined, we'll handle it separately
            taxa_file = os.path.join(base_data_dir, 'taxa', f'{cohort_stem}.tsv')
            pathways_file = os.path.join(base_data_dir, 'pathways', f'{cohort_stem}.tsv')
        else:
            raise ValueError(f"Unknown data_type: {data_type}")
        
        metadata_file = os.path.join(base_data_dir, 'metadata', f'{cohort_stem}.tsv')
        
        # Load metadata
        meta = pd.read_csv(metadata_file, sep="\t", index_col=0)
        if 'Study condition' not in meta.columns:
            print(f"  Warning: '{cohort_stem}' metadata missing 'Study condition' column. Skipping.")
            return None, None
        y_label = meta['Study condition']
        
        # Load and process data based on type
        if data_type == 'combined':
            # Load taxonomy
            species = pd.read_csv(taxa_file, sep="\t", index_col=0)
            species_to_drop = [idx for idx in species.index if "s__" not in idx or "t__" in idx]
            species = species.drop(species_to_drop)
            species.index = [idx.split("s__")[1] for idx in species.index]
            
            # Load pathways
            paths = pd.read_csv(pathways_file, sep="\t", index_col=0)
            
            # Normalize function names: replace spaces with underscores
            # This ensures consistent matching with MDFS output
            paths.index = paths.index.str.replace(' ', '_')
            
            # Normalize both
            species = species.div(species.sum(axis=0), axis=1)
            paths = paths.div(paths.sum(axis=0), axis=1)
            
            # Combine features
            abundance = pd.concat([species, paths], axis=0)
        else:
            # Load abundance matrix
            abundance = pd.read_csv(input_file, sep="\t", index_col=0)
            
            # For taxa, filter species
            if data_type == 'taxa':
                species_to_drop = [idx for idx in abundance.index if "s__" not in idx or "t__" in idx]
                abundance = abundance.drop(species_to_drop)
                abundance.index = [idx.split("s__")[1] for idx in abundance.index]
            elif data_type == 'functions':
                # Normalize function names: replace spaces with underscores
                # This ensures consistent matching with MDFS output
                abundance.index = abundance.index.str.replace(' ', '_')
        
        # Transpose to get samples x features
        X = abundance.T
        
        # Align samples
        common_samples = X.index.intersection(y_label.index)
        X = X.loc[common_samples]
        y_label = y_label.loc[common_samples]
        
        if len(common_samples) == 0:
            print(f"  Warning: '{cohort_stem}' has no common samples. Skipping.")
            return None, None
        
        # Filter out Adenoma
        mask = y_label != 'Adenoma'
        X = X[mask]
        y_label = y_label[mask]
        
        # Encode labels
        y = y_label.replace({'CRC': 0, 'Control': 1}).astype(int)
        
        # Normalize to relative abundance
        X = X.div(X.sum(axis=1), axis=0).fillna(0)
        
        print(f"  Successfully prepared '{cohort_stem}': {len(X)} samples, {X.shape[1]} features")
        return X, y
    except Exception as e:
        print(f"  Error processing '{cohort_stem}': {e}. Skipping.")
        return None, None

def prepare_all_cohorts(data_type='functions', output_dir='prepared_data', base_data_dir='../../data', min_samples_per_class=15):
    """
    Prepares all cohorts and saves them individually.
    
    Parameters:
    -----------
    data_type : str
        Type of data: 'functions', 'taxa', or 'combined'
    output_dir : str
        Directory to save prepared data files
    base_data_dir : str
        Base directory containing data subdirectories
    min_samples_per_class : int
        Minimum samples per class required for a cohort to be eligible for testing
    
    Returns:
    --------
    dict : Dictionary mapping cohort_stem to (X_path, y_path) tuples
    """
    print("="*80)
    print(f"Preparing All Cohorts - {data_type.upper()}")
    print("="*80)
    
    os.makedirs(output_dir, exist_ok=True)
    
    all_prepared = {}
    test_eligible = []
    
    print(f"\n--- Processing {len(ALL_COHORT_STEMS)} cohorts ---")
    for cohort_stem in ALL_COHORT_STEMS:
        print(f"\nProcessing: {cohort_stem}")
        X, y = prepare_single_cohort(cohort_stem, data_type, base_data_dir)
        
        if X is not None and y is not None:
            # Check eligibility for testing
            class_counts = y.value_counts()
            is_eligible = (class_counts.get(0, 0) >= min_samples_per_class and 
                          class_counts.get(1, 0) >= min_samples_per_class)
            
            # Save prepared data
            X_path = os.path.join(output_dir, f"X_{cohort_stem}_{data_type}.tsv")
            y_path = os.path.join(output_dir, f"y_{cohort_stem}_{data_type}.tsv")
            
            X.to_csv(X_path, sep="\t")
            y.to_csv(y_path, sep="\t", header=True)
            
            all_prepared[cohort_stem] = (X_path, y_path)
            
            if cohort_stem == TRAINING_ONLY_COHORT:
                print(f"  Designated as training-only cohort")
            elif is_eligible:
                test_eligible.append(cohort_stem)
                print(f"  ELIGIBLE for testing (CRC={class_counts.get(0,0)}, Control={class_counts.get(1,0)})")
            else:
                print(f"  INELIGIBLE for testing (CRC={class_counts.get(0,0)}, Control={class_counts.get(1,0)})")
    
    print("\n" + "="*80)
    print("Data Preparation Complete!")
    print("="*80)
    print(f"\nSummary:")
    print(f"  Total cohorts processed: {len(all_prepared)}")
    print(f"  Test-eligible cohorts: {len(test_eligible)}")
    print(f"  Training-only cohort: {TRAINING_ONLY_COHORT}")
    print(f"\nTest-eligible cohorts:")
    for cohort in test_eligible:
        print(f"  - {cohort}")
    
    return all_prepared, test_eligible

In [3]:
# Prepare all functions data:
all_prepared, test_eligible = prepare_all_cohorts(
     data_type='functions',
     output_dir='prepared_data',
     base_data_dir='../data',
     min_samples_per_class=15
 )

# Prepare all taxa data:
all_prepared, test_eligible = prepare_all_cohorts(
     data_type='taxa',
     output_dir='prepared_data',
     base_data_dir='../data',
     min_samples_per_class=15
 )

# Prepare all combined data:
all_prepared, test_eligible = prepare_all_cohorts(
     data_type='combined',
     output_dir='prepared_data',
     base_data_dir='../data',
     min_samples_per_class=15
 )

Preparing All Cohorts - FUNCTIONS

--- Processing 17 cohorts ---

Processing: Public_study__FengQ_2015
  Successfully prepared 'Public_study__FengQ_2015': 107 samples, 529 features
  ELIGIBLE for testing (CRC=46, Control=61)

Processing: Public_study__GuptaA_2019
  Successfully prepared 'Public_study__GuptaA_2019': 60 samples, 467 features
  ELIGIBLE for testing (CRC=30, Control=30)

Processing: Public_study__LiuNN_2022
  Successfully prepared 'Public_study__LiuNN_2022': 163 samples, 566 features
  ELIGIBLE for testing (CRC=78, Control=85)

Processing: Public_study__ThomasAM_2018b
  Successfully prepared 'Public_study__ThomasAM_2018b': 60 samples, 481 features
  ELIGIBLE for testing (CRC=32, Control=28)

Processing: Public_study__VogtmannE_2016
  Successfully prepared 'Public_study__VogtmannE_2016': 110 samples, 497 features
  ELIGIBLE for testing (CRC=52, Control=58)

Processing: Public_study__WirbelJ_2018
  Successfully prepared 'Public_study__WirbelJ_2018': 125 samples, 511 features